# OpenClaw x ArangoDB — Enron Demo

Seeds the digital brain with the public Enron email corpus, builds an executive
knowledge graph, and walks through semantic search, session replay, graph
traversal, compaction, and an interactive dashboard.

**Prerequisites:** copy `.env.example` to `.env` and fill in your ArangoDB credentials.

## 1. Connect

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from openclaw_brain import connect
from rich.console import Console
from rich.table import Table

console = Console()
brain = connect()
console.print('[bold green]\u2713 DigitalBrain ready[/bold green]')
console.print(brain.stats())

## 2. Seed — Enron Email Dataset

Downloads a sample of the public Enron email corpus and loads it as memories.
Emails become `conversation` memories; people become entities; org relationships become edges.

In [ ]:
import re
from datasets import load_dataset

console.print('[bold]Loading Enron email dataset...[/bold]')
ds = load_dataset('SetFit/enron_spam', split='train', trust_remote_code=True)
console.print(f'[green]\u2713[/green] Loaded {len(ds)} emails')

def extract_emails(text):
    return list(set(re.findall(r'[\w.+-]+@[\w.-]+\.\w+', str(text or ''))))

def clean_body(text):
    text = re.sub(r'-{3,}.*?(?=\n\n|$)', '', str(text or ''), flags=re.DOTALL)
    text = re.sub(r'(From|To|Subject|Date|Cc):.*?\n', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:400]

SAMPLE = 120
loaded, skipped = 0, 0

for row in ds.select(range(min(SAMPLE * 3, len(ds)))):
    if loaded >= SAMPLE:
        break
    subject = str(row.get('subject') or row.get('Subject') or 'No Subject')[:120]
    body = clean_body(row.get('message') or row.get('body') or row.get('text') or '')
    sender = str(row.get('sender') or row.get('From') or '')[:80]
    label = row.get('label', 0)
    if not body or len(body) < 30:
        skipped += 1
        continue
    content = f'[{subject}] {body}'
    tags = ['enron', 'email', 'spam' if label else 'ham']
    if sender:
        tags.append(sender.split('@')[0][:20])
    ents = extract_emails(sender) + extract_emails(body)
    ents = [e for e in ents if 'enron' in e.lower() or len(e) < 40][:5]
    brain.store(content, memory_type='conversation', tags=tags,
                source='enron_dataset', entities_mentioned=ents[:3])
    loaded += 1

console.print(f'[green]\u2713[/green] Loaded {loaded} emails ({skipped} skipped)')

In [ ]:
console.print('[bold]Building Enron entity graph...[/bold]')

enron_people = [
    ('Kenneth Lay',      'CEO'),
    ('Jeffrey Skilling', 'COO'),
    ('Andrew Fastow',    'CFO'),
    ('Sherron Watkins',  'VP Accounting'),
    ('Lou Pai',          'Chairman EES'),
    ('Rebecca Mark',     'VP International'),
]
enron_rels = [
    ('Kenneth Lay',      'Jeffrey Skilling', 'reports_to'),
    ('Jeffrey Skilling', 'Andrew Fastow',    'manages'),
    ('Andrew Fastow',    'Enron',            'works_at'),
    ('Kenneth Lay',      'Enron',            'leads'),
    ('Sherron Watkins',  'Kenneth Lay',      'reported_fraud_to'),
    ('Sherron Watkins',  'Enron',            'works_at'),
    ('Lou Pai',          'Enron',            'works_at'),
    ('Rebecca Mark',     'Enron',            'works_at'),
    ('Enron',            'Arthur Andersen',  'audited_by'),
    ('Enron',            'SEC',              'investigated_by'),
]
enron_facts = [
    ('Enron Corporation was an American energy company based in Houston, Texas.', 'fact', ['enron','company']),
    ('Enron filed for bankruptcy in December 2001 \u2014 one of the largest in US history.', 'event', ['enron','bankruptcy']),
    ('The Enron scandal involved systematic accounting fraud and off-balance-sheet entities.', 'fact', ['enron','fraud','accounting']),
    ('Kenneth Lay was Enron CEO and was convicted of fraud and conspiracy in 2006.', 'fact', ['enron','executives','fraud']),
    ('Jeffrey Skilling was COO/CEO of Enron and received a 24-year prison sentence.', 'fact', ['enron','executives','fraud']),
    ('Andrew Fastow was CFO and created the off-balance-sheet partnerships that hid debt.', 'fact', ['enron','executives','fraud']),
    ('Sherron Watkins was the Enron whistleblower who warned Kenneth Lay of accounting irregularities.', 'fact', ['enron','whistleblower']),
    ("Arthur Andersen, Enron's auditor, was convicted of obstruction of justice.", 'event', ['enron','andersen','auditor']),
    ("Enron's stock peaked at $90.75 in mid-2000 and collapsed to under $1 by late 2001.", 'fact', ['enron','stock','collapse']),
    ('The Enron email dataset contains ~500,000 emails from 150 employees made public by the FERC.', 'fact', ['enron','dataset','emails']),
]

for content, mtype, tags in enron_facts:
    brain.store(content, memory_type=mtype, tags=tags, source='enron_public_record')

for person, title in enron_people:
    brain.store(f'{person} was {title} at Enron Corporation.',
                memory_type='fact', tags=['enron', 'executive'],
                entities_mentioned=[person, 'Enron'])

for a, b, rel in enron_rels:
    brain.link_entities(a, b, rel)

console.print(f'[green]\u2713[/green] Enron knowledge graph built: {len(enron_people)} people, {len(enron_rels)} relations')
console.print(brain.stats())

## 3. Semantic Search

In [ ]:
def pretty_search(query, top_k=5, memory_type=None):
    results = brain.search(query, top_k=top_k, memory_type=memory_type)
    table = Table(title=f'memory_search: "{query}"', show_lines=True)
    table.add_column('Score',   style='cyan',   width=8)
    table.add_column('Type',    style='yellow', width=14)
    table.add_column('Content', style='white')
    table.add_column('Tags',    style='dim',    width=22)
    for r in results:
        s = r.get('score') or 0
        table.add_row(f'{s:.3f}', r.get('memory_type',''), r.get('content','')[:110],
                      ', '.join(r.get('tags',[]))[:22])
    console.print(table)
    return results

pretty_search('accounting fraud')
pretty_search('bankruptcy 2001')
pretty_search('email communication', memory_type='conversation')

## 4. Session Replay

In [ ]:
import uuid

session_id = str(uuid.uuid4())[:8]
brain.open_session(session_id, agent_id='default', channel='whatsapp')
console.print(f'[bold]Opened session:[/bold] {session_id}')

convo = [
    ('user',      'Tell me about the Enron scandal.'),
    ('assistant', 'Enron was an energy company that collapsed in 2001 due to systematic accounting fraud. '
                  'Executives including Kenneth Lay, Jeffrey Skilling, and Andrew Fastow were convicted.'),
    ('user',      'Who was the whistleblower?'),
    ('assistant', 'Sherron Watkins was the Enron whistleblower. She sent a memo to Kenneth Lay '
                  'warning of accounting irregularities in August 2001.'),
    ('user',      'Remember: I need to review the Arthur Andersen audit trail.'),
    ('assistant', 'Noted \u2014 review the Arthur Andersen audit trail for Enron.'),
]

for role, content in convo:
    brain.store_message(session_id, role=role, content=content)

brain.store('Review Arthur Andersen audit trail and Enron SPE documentation.',
            memory_type='decision', tags=['action','enron','audit'], session_id=session_id)

console.print(f'[green]\u2713[/green] Stored {len(convo)} messages + 1 durable fact')
pretty_search('Arthur Andersen audit', top_k=3)

## 5. Entity Knowledge Graph

In [ ]:
console.print('[bold]Entity neighbourhood: Enron (depth=2)[/bold]')
neighbours = brain.entity_neighbourhood('Enron', depth=2)

table = Table(title='Graph Neighbourhood', show_lines=True)
table.add_column('Depth',    width=6)
table.add_column('Relation', style='cyan', width=20)
table.add_column('Entity / Memory', style='white')
for n in neighbours[:20]:
    v = n['vertex']
    label = v.get('name') or v.get('content','')[:70]
    table.add_row(str(n['depth']), n.get('edge_label',''), label)
console.print(table)

## 6. Heartbeat Compaction

In [ ]:
from datetime import date

daily = brain.compact_day(target_date=date.today())
if daily:
    console.print(f'[green]\u2713[/green] Compacted {daily["entry_count"]} memories \u2192 daily log: {daily["date"]}')
    console.print(f'\n[bold]Daily Summary ({daily["date"]}):[/bold]')
    print(daily['summary'])
else:
    console.print('[yellow]No memories to compact for today[/yellow]')

## 7. Tool Shim Test

In [ ]:
from openclaw_brain.tools import memory_store, memory_search, memory_get, memory_delete

console.print('[bold]Testing tool shim...[/bold]')
r = memory_store(brain, 'Enron filed for Chapter 11 bankruptcy on December 2nd 2001.',
                 memory_type='event', tags=['enron','bankruptcy'])
console.print(f'  memory_store  -> {r}')

hits = memory_search(brain, 'Enron bankruptcy filing', top_k=3)
console.print(f'  memory_search -> {len(hits)} result(s)')
for h in hits:
    console.print(f'    [{h["score"]:.3f}] {h["content"][:80]}')

if hits:
    got = memory_get(brain, hits[0]['source'].replace('memories/', ''))
    console.print(f'  memory_get    -> {got["text"][:70]}...')

console.print('[bold green]\u2713 Tool shim working[/bold green]')

## 8. AQL Brain Inspector

In [ ]:
db = brain.db

def aql(query, bind_vars=None):
    return list(db.aql.execute(query, bind_vars=bind_vars or {}))

type_counts = aql('FOR m IN memories COLLECT t = m.memory_type WITH COUNT INTO n SORT n DESC RETURN {type: t, count: n}')
table = Table(title='Memory Inventory')
table.add_column('Type', style='yellow')
table.add_column('Count', style='cyan', justify='right')
for row in type_counts:
    table.add_row(row['type'], str(row['count']))
console.print(table)

rels = aql('FOR e IN entity_edges LET f = DOCUMENT(e._from) LET t = DOCUMENT(e._to) RETURN {from: f.name, rel: e.relation, to: t.name || LEFT(t.content, 50)}')
table2 = Table(title='Entity Knowledge Graph', show_lines=True)
table2.add_column('From', style='green')
table2.add_column('Relation', style='cyan')
table2.add_column('To', style='blue')
for row in rels[:25]:
    table2.add_row(row.get('from','?'), row.get('rel',''), row.get('to','?'))
console.print(table2)

top = aql('FOR m IN memories SORT m.access_count DESC LIMIT 5 RETURN {accesses: m.access_count, type: m.memory_type, content: LEFT(m.content,80)}')
table3 = Table(title='Most Accessed Memories', show_lines=True)
table3.add_column('Hits', style='cyan', width=6)
table3.add_column('Type', style='yellow', width=14)
table3.add_column('Content')
for row in top:
    table3.add_row(str(row['accesses']), row['type'], row['content'])
console.print(table3)

## 9. SKILL.md Generator

In [ ]:
import os

host = os.environ.get('ARANGO_HOST', '???')
db_name = os.environ.get('ARANGO_DB_NAME', 'openclaw_brain')

skill_md = f"""---
name: arango-brain
description: ArangoDB-backed digital brain - replaces SQLite memory with graph + vector + document storage
version: 1.0.0
---

# ArangoDB Digital Brain

Replaces OpenClaw's default SQLite memory layer with ArangoDB:
graph traversal, vector search, document storage, entity linking, and daily compaction.

## Connection
- Host:     {host}
- Database: {db_name}

## Tools

### memory_store(content, memory_type, tags)
Store a durable memory. Call when:
- User says 'remember', 'note that', 'don't forget'
- A decision, preference, or key fact arises
- You want to persist something across sessions

memory_type: fact | event | note | conversation | decision | preference | daily_summary

### memory_search(query, top_k, memory_type)
Semantic search over all memories. Returns ranked list with source paths.
Always call this before saying 'I don't know' about something personal.

### memory_get(key)
Fetch a specific memory by key (from memory_search source field).

### memory_delete(key)
Remove a memory by key.

## Memory types guide
| Type | Use for |
|------|--------|
| fact | Timeless facts about user, world, projects |
| event | Things that happened at a point in time |
| decision | Architectural, personal or strategic choices |
| preference | User preferences and settings |
| note | General reminders and to-dos |
| conversation | Session message turns |
| daily_summary | Auto-generated daily compaction entries |

## Source citation
memory_search results include source: memories/<key>.
You can reference this as Source: memories/<key> in responses.
"""

with open('/tmp/SKILL_arango_brain.md', 'w') as f:
    f.write(skill_md)
console.print('[green]\u2713[/green] SKILL.md written to /tmp/SKILL_arango_brain.md')
print(skill_md)

## 10. Health Check

In [ ]:
console.print('[bold]\u2500\u2500\u2500 OpenClaw \u00d7 ArangoDB Brain \u2014 Health Check \u2500\u2500\u2500[/bold]')
hc = brain.health_check()
for label, ok in hc.items():
    if label == 'counts':
        continue
    icon = '[green]\u2713[/green]' if ok else '[yellow]\u25cb[/yellow]'
    console.print(f'  {icon} {label}')
console.print('\n[bold]Collection counts:[/bold]')
for k, v in hc['counts'].items():
    console.print(f'  {k:<16} {v}')
console.print('\n[bold green]OpenClaw Digital Brain is live on ArangoDB[/bold green]')

## 11. Interactive Dashboard

Renders a full HTML/d3 dashboard in an iframe. Click **Open Brain UI** to launch it in a new tab.

In [ ]:
from IPython.display import display, HTML as _IHTML
import json as _json, base64 as _b64

_mem = list(db.aql.execute(
    "FOR m IN memories SORT m.created_at DESC LIMIT 200 "
    "RETURN {_key:m._key,content:m.content,type:m.memory_type,"
    "tags:m.tags,created:m.created_at,access:m.access_count}"
))
_ent = list(db.aql.execute(
    "FOR e IN entity_edges "
    "LET f=DOCUMENT(e._from) LET t=DOCUMENT(e._to) "
    "RETURN {fn:f.name,tn:t.name||LEFT(t.content,40),rel:e.relation}"
))
_st = brain.stats()
_bt = list(db.aql.execute(
    "FOR m IN memories COLLECT t=m.memory_type WITH COUNT INTO n "
    "SORT n DESC RETURN {type:t,count:n}"
))
_tr = list(db.aql.execute(
    "FOR m IN memories LET d=LEFT(m.created_at,10) "
    "COLLECT day=d WITH COUNT INTO n SORT day DESC LIMIT 7 RETURN {day:day,count:n}"
))
_DATA = _json.dumps({
    "memories":_mem,"entities":_ent,
    "stats":{"totals":_st,"by_type":_bt,"trend":_tr}
}, ensure_ascii=True)

# The full dashboard HTML template is loaded from the original notebook's
# embedded UI (d3 force graph, search, store, stats panels).
# For brevity it is kept as a single-line string; open in a browser for
# the best experience.
_T = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8"/>\n<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=JetBrains+Mono:wght@300;400;500&display=swap" rel="stylesheet"/>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>\n<style>\n:root{--bg:#090d12;--bg1:#0e1318;--bg2:#141b24;--bg3:#1a2535;--bo:#1d2b3a;--bo2:#28394f;--ac:#00c17c;--a2:#0088ee;--a3:#e84d2a;--a4:#f5a623;--tx:#dce8f2;--tx2:#7a8fa8;--tx3:#3d5068;--mo:\'JetBrains Mono\',monospace;--sa:\'Syne\',sans-serif}\n*{box-sizing:border-box;margin:0;padding:0}\nbody{background:var(--bg);color:var(--tx);font-family:var(--sa);min-height:100vh}\n::-webkit-scrollbar{width:4px}::-webkit-scrollbar-track{background:transparent}::-webkit-scrollbar-thumb{background:var(--bo2);border-radius:4px}\n\n.hdr{display:flex;align-items:center;justify-content:space-between;padding:14px 24px;background:var(--bg1);border-bottom:1px solid var(--bo);position:sticky;top:0;z-index:50}\n.brand{display:flex;align-items:center;gap:10px}\n.brand-ico{font-size:20px;animation:glow 3s ease-in-out infinite}\n@keyframes glow{0%,100%{filter:drop-shadow(0 0 4px #00c17c50)}50%{filter:drop-shadow(0 0 12px #00c17c)}}\n.brand-name{font-size:14px;font-weight:800;letter-spacing:.04em}\n.brand-name b{color:var(--ac)}\n.brand-sub{font-size:9px;color:var(--tx3);font-family:var(--mo);letter-spacing:.12em}\n.hstats{display:flex;gap:20px}\n.hs{text-align:right}\n.hs-v{font-family:var(--mo);font-size:18px;color:var(--ac);line-height:1}\n.hs-l{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em;margin-top:2px}\n.nav{display:flex;padding:0 24px;background:var(--bg1);border-bottom:1px solid var(--bo)}\n.tab{padding:11px 16px;font-size:11px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;cursor:pointer;border:none;background:none;color:var(--tx3);border-bottom:2px solid transparent;transition:all .2s}\n.tab:hover{color:var(--tx2)}\n.tab.on{color:var(--ac);border-bottom-color:var(--ac)}\n.main{padding:20px 24px}\n.panel{display:none}.panel.on{display:block}\n.card{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;padding:16px;margin-bottom:14px}\n.ctitle{font-size:10px;font-weight:700;letter-spacing:.1em;text-transform:uppercase;color:var(--tx3);margin-bottom:14px;display:flex;align-items:center;gap:6px}\n.ctitle::before{content:\'\';display:inline-block;width:2px;height:10px;background:var(--ac);border-radius:2px}\n.sbar{display:flex;gap:8px;background:var(--bg2);border:1px solid var(--bo2);border-radius:6px;padding:3px 3px 3px 14px;margin-bottom:12px;transition:border-color .2s;align-items:center}\n.sbar:focus-within{border-color:var(--ac)}\n.sinp{flex:1;background:none;border:none;outline:none;color:var(--tx);font-family:var(--mo);font-size:13px;padding:9px 0}\n.sinp::placeholder{color:var(--tx3)}\n.btn{padding:9px 16px;border:none;border-radius:5px;cursor:pointer;font-family:var(--sa);font-size:11px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;transition:all .15s}\n.btn-g{background:var(--ac);color:#000}.btn-g:hover{background:#00e090}\n.btn-r{background:transparent;border:1px solid var(--a3);color:var(--a3)}.btn-r:hover{background:var(--a3);color:#fff}\n.chips{display:flex;gap:6px;flex-wrap:wrap;align-items:center;margin-bottom:12px}\n.chip{padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;cursor:pointer;border:1px solid var(--bo2);background:var(--bg2);color:var(--tx3);transition:all .15s}\n.chip:hover,.chip.on{background:var(--ac);color:#000;border-color:var(--ac)}\n.rgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:10px;margin-top:4px}\n.mc{background:var(--bg2);border:1px solid var(--bo);border-radius:6px;padding:14px;cursor:pointer;transition:all .2s;position:relative;overflow:hidden}\n.mc::before{content:\'\';position:absolute;left:0;top:0;bottom:0;width:3px;background:var(--ac);opacity:0;transition:opacity .2s}\n.mc:hover{border-color:var(--bo2);transform:translateY(-1px);box-shadow:0 4px 16px #00000040}\n.mc:hover::before,.mc.sel::before{opacity:1}\n.mc.sel{border-color:var(--ac);background:var(--bg3)}\n.mt{display:inline-block;padding:2px 8px;border-radius:3px;font-size:9px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;margin-bottom:8px}\n.mc-txt{font-family:var(--mo);font-size:11px;color:var(--tx);line-height:1.65;margin-bottom:8px}\n.mc-meta{display:flex;gap:7px;align-items:center;flex-wrap:wrap}\n.mc-score{font-family:var(--mo);font-size:10px;padding:2px 6px;background:var(--bg3);border-radius:3px;color:var(--ac)}\n.mc-tag{font-size:9px;padding:2px 6px;border-radius:3px;background:var(--bg3);color:var(--tx3);border:1px solid var(--bo)}\n.mc-date{font-size:9px;color:var(--tx3);font-family:var(--mo);margin-left:auto}\n.mc-key{font-size:8px;color:var(--tx3);font-family:var(--mo);margin-top:4px;opacity:.4}\n.nores{text-align:center;padding:50px 20px;color:var(--tx3);font-family:var(--mo);font-size:12px;grid-column:1/-1}\n.t-fact{background:#00c17c18;color:#00c17c}\n.t-event{background:#0088ee18;color:#5599ff}\n.t-decision{background:#e84d2a18;color:#e84d2a}\n.t-preference{background:#f5a62318;color:#f5a623}\n.t-note{background:#7a8fa818;color:#7a8fa8}\n.t-conversation{background:#aa44ff18;color:#aa44ff}\n.t-daily_summary{background:#00c17c10;color:#00c17c}\n.dpane{position:fixed;right:0;top:0;bottom:0;width:380px;background:var(--bg1);border-left:1px solid var(--bo);padding:20px;overflow-y:auto;transform:translateX(100%);transition:transform .25s cubic-bezier(.4,0,.2,1);z-index:200}\n.dpane.open{transform:translateX(0)}\n.dclose{position:absolute;top:12px;right:12px;background:var(--bg2);border:1px solid var(--bo);color:var(--tx2);cursor:pointer;font-size:13px;padding:5px 9px;border-radius:4px}\n.dfield{margin-bottom:12px}\n.dfl{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em;margin-bottom:4px}\n.dfv{font-family:var(--mo);font-size:11px;color:var(--tx);background:var(--bg2);padding:9px 11px;border-radius:4px;line-height:1.6;border:1px solid var(--bo);word-break:break-word}\n.sgrid2{display:grid;grid-template-columns:1fr 270px;gap:14px;align-items:start}\n.inp,.sel,.ta{width:100%;background:var(--bg2);border:1px solid var(--bo2);border-radius:6px;color:var(--tx);font-family:var(--mo);font-size:13px;padding:10px 12px;outline:none;transition:border-color .2s}\n.inp:focus,.sel:focus,.ta:focus{border-color:var(--ac)}\n.ta{min-height:100px;resize:vertical;line-height:1.6}\n.sel option{background:var(--bg2)}\n.lbl{display:block;font-size:10px;color:var(--tx3);text-transform:uppercase;letter-spacing:.08em;margin-bottom:6px}\n.fg{margin-bottom:12px}\n.rlist{display:flex;flex-direction:column;gap:7px}\n.ri{padding:9px 11px;background:var(--bg2);border-radius:4px;border:1px solid var(--bo)}\n.ric{font-size:11px;font-family:var(--mo);color:var(--tx);line-height:1.5;margin-bottom:3px;overflow:hidden;display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical}\n.rim{font-size:9px;color:var(--tx3)}\n.kpis{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:16px}\n.kpi{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;padding:16px;text-align:center;position:relative;overflow:hidden}\n.kpi::after{content:\'\';position:absolute;bottom:0;left:0;right:0;height:2px;background:linear-gradient(90deg,var(--ac),var(--a2))}\n.knum{font-family:var(--mo);font-size:26px;font-weight:500;color:var(--ac);line-height:1;margin-bottom:4px}\n.klbl{font-size:9px;color:var(--tx3);text-transform:uppercase;letter-spacing:.1em}\n.charts{display:grid;grid-template-columns:1fr 1fr;gap:14px}\n.bi{display:flex;align-items:center;gap:8px;margin-bottom:7px}\n.bl{font-size:10px;color:var(--tx2);width:110px;font-family:var(--mo);text-align:right;flex-shrink:0}\n.btr{flex:1;background:var(--bg3);border-radius:2px;height:7px;overflow:hidden}\n.bf{height:100%;border-radius:2px;background:linear-gradient(90deg,var(--ac),var(--a2));transition:width 1s cubic-bezier(.4,0,.2,1)}\n.bc{font-size:10px;color:var(--tx3);font-family:var(--mo);width:24px;text-align:right}\n.tbars{display:flex;align-items:flex-end;gap:5px;height:100px}\n.tbar{flex:1;border-radius:3px 3px 0 0;background:linear-gradient(180deg,var(--ac),var(--a2));min-height:3px}\n.tbar:hover{filter:brightness(1.3)}\n.tlbls{display:flex;gap:5px;margin-top:5px}\n.tlbl{flex:1;text-align:center;font-size:8px;color:var(--tx3);font-family:var(--mo)}\n.gwrap{background:var(--bg1);border:1px solid var(--bo);border-radius:6px;overflow:hidden;position:relative}\n.gctrl{position:absolute;top:10px;left:10px;z-index:10;display:flex;flex-direction:column;gap:5px}\n.gbtn{width:30px;height:30px;display:flex;align-items:center;justify-content:center;background:var(--bg2);border:1px solid var(--bo2);border-radius:4px;cursor:pointer;font-size:14px;color:var(--tx2);user-select:none}\n.gbtn:hover{background:var(--bg3);color:var(--tx)}\n.gleg{position:absolute;bottom:10px;left:10px;background:var(--bg2);border:1px solid var(--bo);border-radius:4px;padding:8px 10px}\n.gli{display:flex;align-items:center;gap:6px;font-size:10px;color:var(--tx2);margin-bottom:2px}\n.gli:last-child{margin:0}\n.gld{width:8px;height:8px;border-radius:50%}\n.gtip{position:fixed;padding:6px 10px;background:var(--bg3);border:1px solid var(--bo2);border-radius:4px;font-size:11px;font-family:var(--mo);color:var(--tx);pointer-events:none;z-index:300;display:none;max-width:180px}\n.toast{position:fixed;bottom:20px;right:20px;padding:10px 16px;background:var(--ac);color:#000;border-radius:6px;font-weight:700;font-size:12px;z-index:999;animation:sup .2s ease,fout .3s ease 1.7s forwards;pointer-events:none}\n.toast.err{background:var(--a3);color:#fff}\n@keyframes sup{from{transform:translateY(12px);opacity:0}to{transform:translateY(0);opacity:1}}\n@keyframes fout{to{opacity:0}}\n@media(max-width:800px){.sgrid2{grid-template-columns:1fr}.charts{grid-template-columns:1fr}.kpis{grid-template-columns:1fr 1fr}.rgrid{grid-template-columns:1fr}}\n</style>\n</head>\n<body>\n<div class="hdr"><div class="brand"><span class="brand-ico">&#x1F99E;</span><div><div class="brand-name">OpenClaw <b>x</b> ArangoDB</div><div class="brand-sub">DIGITAL BRAIN</div></div></div><div class="hstats"><div class="hs"><div class="hs-v" id="hm">0</div><div class="hs-l">Memories</div></div><div class="hs"><div class="hs-v" id="he">0</div><div class="hs-l">Entities</div></div><div class="hs"><div class="hs-v" id="hg">0</div><div class="hs-l">Edges</div></div><div class="hs"><div class="hs-v" id="hs2">0</div><div class="hs-l">Sessions</div></div></div></div>\n<div class="nav"><button class="tab on" onclick="go(\'search\',this)">&#128269; Search</button><button class="tab" onclick="go(\'store\',this)">&#9999; Store</button><button class="tab" onclick="go(\'graph\',this)">&#128375; Graph</button><button class="tab" onclick="go(\'stats\',this)">&#128202; Stats</button></div>\n<div class="main"><div id="p-search" class="panel on"><div class="sbar"><input class="sinp" id="sinp" placeholder="Search memories..." onkeydown="if(event.key===\'Enter\')doSearch()"/><button class="btn btn-g" onclick="doSearch()">Search</button></div><div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;flex-wrap:wrap;gap:8px"><div class="chips" style="margin:0" id="tchips"><span class="chip on" onclick="ftype(this,\'\')">All</span><span class="chip" onclick="ftype(this,\'fact\')">Fact</span><span class="chip" onclick="ftype(this,\'event\')">Event</span><span class="chip" onclick="ftype(this,\'decision\')">Decision</span><span class="chip" onclick="ftype(this,\'preference\')">Preference</span><span class="chip" onclick="ftype(this,\'note\')">Note</span><span class="chip" onclick="ftype(this,\'conversation\')">Conv</span></div><select class="sel" id="topk" style="width:80px;padding:6px 8px;font-size:11px" onchange="doSearch()"><option>5</option><option selected>8</option><option>15</option><option>30</option></select></div><div id="sres" class="rgrid"></div></div><div id="p-store" class="panel"><div class="sgrid2"><div class="card"><div class="ctitle">New Memory</div><div class="fg"><label class="lbl">Content</label><textarea class="ta" id="sc" placeholder="What should the brain remember?"></textarea></div><div style="display:grid;grid-template-columns:1fr 1fr;gap:10px"><div class="fg"><label class="lbl">Type</label><select class="sel" id="stype"><option value="fact">Fact</option><option value="event">Event</option><option value="decision">Decision</option><option value="preference">Preference</option><option value="note">Note</option><option value="conversation">Conversation</option></select></div><div class="fg"><label class="lbl">Tags (comma-sep)</label><input class="inp" id="stags" placeholder="tag1, tag2..."/></div></div><button class="btn btn-g" style="width:100%;padding:11px" onclick="doStore()">Store Memory</button></div><div class="card"><div class="ctitle">Recent Memories</div><div class="rlist" id="rlist"></div></div></div></div><div id="p-graph" class="panel"><div class="ctitle" style="margin-bottom:10px">Entity Knowledge Graph</div><div class="gwrap" id="gwrap"><div class="gctrl"><div class="gbtn" onclick="gzi()">+</div><div class="gbtn" onclick="gzo()">-</div><div class="gbtn" onclick="gzr()">o</div></div><svg id="gsvg"></svg><div class="gleg"><div class="gli"><div class="gld" style="background:#00c17c"></div>Entity</div><div class="gli"><div class="gld" style="background:#5599ff"></div>Memory ref</div></div></div><div class="gtip" id="gtip"></div></div><div id="p-stats" class="panel"><div class="kpis" id="kpis"></div><div class="charts"><div class="card"><div class="ctitle">By Type</div><div id="barchart"></div></div><div class="card"><div class="ctitle">Last 7 Days</div><div class="tbars" id="trbars"></div><div class="tlbls" id="trlbls"></div></div></div></div></div>\n<div class="dpane" id="dpane"><button class="dclose" onclick="document.getElementById(\'dpane\').classList.remove(\'open\')">x</button><div id="dcont"></div></div>\n<script type="application/json" id="__braindata__">__JSONDATA__</script>\n<script>var D=JSON.parse(document.getElementById(\'__braindata__\').textContent);var curType=\'\',selKey=null,zb=null;var t=D.stats.totals;document.getElementById(\'hm\').textContent=t.memories||0;document.getElementById(\'he\').textContent=t.entities||0;document.getElementById(\'hg\').textContent=t.entity_edges||0;document.getElementById(\'hs2\').textContent=t.sessions||0;showMems(D.memories.slice(0,8));loadRecent();function go(name,btn){document.querySelectorAll(\'.tab\').forEach(function(t){t.classList.remove(\'on\')});btn.classList.add(\'on\');document.querySelectorAll(\'.panel\').forEach(function(p){p.classList.remove(\'on\')});document.getElementById(\'p-\'+name).classList.add(\'on\');if(name===\'graph\')setTimeout(initGraph,60);if(name===\'stats\')renderStats()}function doSearch(){var q=document.getElementById(\'sinp\').value.trim();var k=parseInt(document.getElementById(\'topk\').value,10);var pool=curType?D.memories.filter(function(m){return m.type===curType}):D.memories;if(!q){showMems(pool.slice(0,k));return}var ql=q.toLowerCase();var words=ql.split(/\\s+/).filter(function(w){return w.length>2});var scored=pool.map(function(m){var c=(m.content||\'\').toLowerCase();var s=0;if(c.indexOf(ql)>-1)s=0.97;else{for(var i=0;i<words.length;i++){if(c.indexOf(words[i])>-1){s=Math.max(s,0.7)}}var tags=m.tags||[];for(var j=0;j<tags.length;j++){if((tags[j]||\'\').toLowerCase().indexOf(ql)>-1)s=Math.max(s,0.6)}}return{_key:m._key,content:m.content,type:m.type,tags:m.tags,created:m.created,access:m.access,score:s}}).filter(function(m){return m.score>0}).sort(function(a,b){return b.score-a.score}).slice(0,k);showMems(scored.length?scored:pool.slice(0,k))}function ftype(el,t){curType=t;document.querySelectorAll(\'.chip\').forEach(function(c){c.classList.remove(\'on\')});el.classList.add(\'on\');doSearch()}function showMems(items){var el=document.getElementById(\'sres\');if(!items||!items.length){el.innerHTML=\'<div class="nores">No memories found.</div>\';return}var html=\'\';for(var i=0;i<items.length;i++){var m=items[i];var key=m._key||\'\';var type=m.type||\'note\';var txt=(m.content||\'\').slice(0,130);if((m.content||\'\').length>130)txt+=\'...\';var score=(m.score>0)?\'<span class="mc-score">\'+m.score.toFixed(3)+\'</span>\':\'\';var tags=\'\';var ta=m.tags||[];for(var j=0;j<Math.min(ta.length,3);j++){tags+=\'<span class="mc-tag">\'+esc(ta[j])+\'</span>\'}var date=fd(m.created||\'\');html+=\'<div class="mc\'+(key===selKey?\' sel\':\'\')+ \'" onclick="selMem(\\\'\'+key+\'\\\',this)">\'+ \'<div class="mt t-\'+type+\'">\'+type+\'</div>\'+ \'<div class="mc-txt">\'+esc(txt)+\'</div>\'+ \'<div class="mc-meta">\'+score+tags+\'<span class="mc-date">\'+date+\'</span></div>\'+ \'<div class="mc-key">\'+key+\'</div>\'+ \'</div>\'}el.innerHTML=html}function selMem(key,el){document.querySelectorAll(\'.mc\').forEach(function(c){c.classList.remove(\'sel\')});el.classList.add(\'sel\');selKey=key;var m=null;for(var i=0;i<D.memories.length;i++){if(D.memories[i]._key===key){m=D.memories[i];break}}if(!m)return;var score=m.score>0?\'<div class="dfield"><div class="dfl">Score</div><div class="dfv">\'+m.score.toFixed(4)+\'</div></div>\':\'\';document.getElementById(\'dcont\').innerHTML=\'<div style="margin-bottom:14px"><span class="mt t-\'+(m.type||\'note\')+\'">\'+( m.type||\'note\')+\'</span></div>\'+\'<div class="dfield"><div class="dfl">Content</div><div class="dfv">\'+esc(m.content||\'\')+ \'</div></div>\'+score+\'<div class="dfield"><div class="dfl">Tags</div><div class="dfv">\'+((m.tags||[]).join(\', \')||\'none\')+\'</div></div>\'+\'<div class="dfield"><div class="dfl">Key</div><div class="dfv">\'+key+\'</div></div>\'+\'<div class="dfield"><div class="dfl">Created</div><div class="dfv">\'+fdf(m.created||\'\')+ \'</div></div>\'+\'<div class="dfield"><div class="dfl">Accesses</div><div class="dfv">\'+( m.access||0)+\'</div></div>\'+\'<div style="margin-top:14px"><button class="btn btn-r" style="width:100%" onclick="doDel(\\\'\'+key+\'\\\')">&gt;Delete</button></div>\';document.getElementById(\'dpane\').classList.add(\'open\')}function doDel(key){if(!key||!confirm(\'Delete?\'))return;for(var i=0;i<D.memories.length;i++){if(D.memories[i]._key===key){D.memories.splice(i,1);break}}document.getElementById(\'dpane\').classList.remove(\'open\');doSearch();document.getElementById(\'hm\').textContent=D.memories.length;toast(\'Deleted\')}function loadRecent(){var html=\'\';var items=D.memories.slice(0,6);for(var i=0;i<items.length;i++){var m=items[i];html+=\'<div class="ri"><div class="ric">\'+esc(m.content||\'\')+ \'</div><div class="rim"><span class="mt t-\'+(m.type||\'note\')+\'" style="font-size:8px;padding:1px 5px">\'+( m.type||\'note\')+\'</span> \'+fd(m.created||\'\')+ \'</div></div>\'}document.getElementById(\'rlist\').innerHTML=html||\'<div style="color:var(--tx3);font-size:11px;font-family:var(--mo)">No memories yet</div>\'}function doStore(){var c=document.getElementById(\'sc\').value.trim();if(!c){toast(\'Content required\',true);return}var ty=document.getElementById(\'stype\').value;var rawTags=document.getElementById(\'stags\').value.split(\',\');var tags=[];for(var i=0;i<rawTags.length;i++){var t2=rawTags[i].trim();if(t2)tags.push(t2)}var m={_key:Math.random().toString(36).slice(2,10),content:c,type:ty,tags:tags,created:new Date().toISOString(),access:0,score:0};D.memories.unshift(m);document.getElementById(\'sc\').value=\'\';document.getElementById(\'stags\').value=\'\';loadRecent();document.getElementById(\'hm\').textContent=D.memories.length;toast(\'Stored!\')}function initGraph(){var wrap=document.getElementById(\'gwrap\');var W=wrap.clientWidth||800;var H=Math.max(500,window.innerHeight*0.6);var svgEl=document.getElementById(\'gsvg\');svgEl.setAttribute(\'width\',W);svgEl.setAttribute(\'height\',H);d3.select(\'#gsvg\').selectAll(\'*\').remove();var svg=d3.select(\'#gsvg\'),g=svg.append(\'g\');zb=d3.zoom().scaleExtent([0.1,5]).on(\'zoom\',function(e){g.attr(\'transform\',e.transform)});svg.call(zb);var nm={};var links=[];for(var i=0;i<D.entities.length;i++){var e=D.entities[i];if(e.fn&&!nm[e.fn])nm[e.fn]={id:e.fn,t:\'entity\'};if(e.tn&&!nm[e.tn])nm[e.tn]={id:e.tn,t:(e.tn.length>30?\'memory\':\'entity\')};if(e.fn&&e.tn)links.push({source:e.fn,target:e.tn,rel:e.rel||\'\'})}var nodes=Object.values(nm);if(!nodes.length){g.append(\'text\').attr(\'x\',W/2).attr(\'y\',H/2).attr(\'text-anchor\',\'middle\').attr(\'fill\',\'#3d5068\').attr(\'font-size\',13).attr(\'font-family\',\'JetBrains Mono,monospace\').text(\'No entity edges to display\');return}svg.append(\'defs\').append(\'marker\').attr(\'id\',\'arr\').attr(\'viewBox\',\'0 -3 7 7\').attr(\'refX\',13).attr(\'markerWidth\',5).attr(\'markerHeight\',5).attr(\'orient\',\'auto\').append(\'path\').attr(\'d\',\'M0,-3L7,0L0,3\').attr(\'fill\',\'#28394f\');var sim=d3.forceSimulation(nodes).force(\'link\',d3.forceLink(links).id(function(d){return d.id}).distance(90)).force(\'charge\',d3.forceManyBody().strength(-200)).force(\'center\',d3.forceCenter(W/2,H/2)).force(\'col\',d3.forceCollide(22));var link=g.append(\'g\').selectAll(\'line\').data(links).join(\'line\').attr(\'stroke\',\'#28394f\').attr(\'stroke-width\',1.5).attr(\'marker-end\',\'url(#arr)\');var llbl=g.append(\'g\').selectAll(\'text\').data(links).join(\'text\').text(function(d){return d.rel}).attr(\'fill\',\'#3d5068\').attr(\'font-size\',8).attr(\'font-family\',\'JetBrains Mono,monospace\').attr(\'text-anchor\',\'middle\');var tip=document.getElementById(\'gtip\');var node=g.append(\'g\').selectAll(\'g\').data(nodes).join(\'g\').call(d3.drag().on(\'start\',function(e,d){if(!e.active)sim.alphaTarget(0.3).restart();d.fx=d.x;d.fy=d.y}).on(\'drag\',function(e,d){d.fx=e.x;d.fy=e.y}).on(\'end\',function(e,d){if(!e.active)sim.alphaTarget(0);d.fx=null;d.fy=null})).on(\'mouseover\',function(e,d){tip.style.display=\'block\';tip.style.left=(e.clientX+10)+\'px\';tip.style.top=(e.clientY-6)+\'px\';tip.textContent=d.id}).on(\'mousemove\',function(e){tip.style.left=(e.clientX+10)+\'px\';tip.style.top=(e.clientY-6)+\'px\'}).on(\'mouseout\',function(){tip.style.display=\'none\'});node.append(\'circle\').attr(\'r\',function(d){return d.t===\'entity\'?12:8}).attr(\'fill\',function(d){return d.t===\'entity\'?\'#00c17c18\':\'#5599ff18\'}).attr(\'stroke\',function(d){return d.t===\'entity\'?\'#00c17c\':\'#5599ff\'}).attr(\'stroke-width\',1.5);node.append(\'text\').text(function(d){return d.id.length>12?d.id.slice(0,12)+\'...\'  :d.id}).attr(\'fill\',function(d){return d.t===\'entity\'?\'#00c17c\':\'#7aabcc\'}).attr(\'font-size\',9).attr(\'font-family\',\'JetBrains Mono,monospace\').attr(\'text-anchor\',\'middle\').attr(\'dy\',\'2.6em\');sim.on(\'tick\',function(){link.attr(\'x1\',function(d){return d.source.x}).attr(\'y1\',function(d){return d.source.y}).attr(\'x2\',function(d){return d.target.x}).attr(\'y2\',function(d){return d.target.y});llbl.attr(\'x\',function(d){return(d.source.x+d.target.x)/2}).attr(\'y\',function(d){return(d.source.y+d.target.y)/2});node.attr(\'transform\',function(d){return\'translate(\'+d.x+\',\'+d.y+\')\'})});setTimeout(function(){sim.alpha(1).restart()},200)}function gzi(){d3.select(\'#gsvg\').transition().call(zb.scaleBy,1.4)}function gzo(){d3.select(\'#gsvg\').transition().call(zb.scaleBy,0.7)}function gzr(){d3.select(\'#gsvg\').transition().call(zb.transform,d3.zoomIdentity)}function renderStats(){var s=D.stats.totals;var rows=[{l:\'Total Memories\',v:s.memories||0},{l:\'Entities\',v:s.entities||0},{l:\'Entity Edges\',v:s.entity_edges||0},{l:\'Memory Edges\',v:s.memory_edges||0},{l:\'Sessions\',v:s.sessions||0},{l:\'Daily Logs\',v:s.daily_logs||0}];var khtml=\'\';for(var i=0;i<rows.length;i++)khtml+=\'<div class="kpi"><div class="knum">\'+rows[i].v+\'</div><div class="klbl">\'+rows[i].l+\'</div></div>\';document.getElementById(\'kpis\').innerHTML=khtml;var bt=D.stats.by_type||[];var mx=1;for(var i=0;i<bt.length;i++)if(bt[i].count>mx)mx=bt[i].count;var bhtml=\'\';for(var i=0;i<bt.length;i++)bhtml+=\'<div class="bi"><div class="bl">\'+bt[i].type+\'</div><div class="btr"><div class="bf" style="width:\'+(bt[i].count/mx*100).toFixed(0)+\'%"></div></div><div class="bc">\'+bt[i].count+\'</div></div>\';document.getElementById(\'barchart\').innerHTML=bhtml;var tr=(D.stats.trend||[]).slice().reverse();var mxT=1;for(var i=0;i<tr.length;i++)if(tr[i].count>mxT)mxT=tr[i].count;var tbhtml=\'\',tlhtml=\'\';for(var i=0;i<tr.length;i++){tbhtml+=\'<div class="tbar" style="height:\'+Math.max(3,(tr[i].count/mxT*95))+\'px" title="\'+tr[i].day+\': \'+tr[i].count+\'"></div>\';tlhtml+=\'<div class="tlbl">\'+tr[i].day.slice(5)+\'</div>\'}document.getElementById(\'trbars\').innerHTML=tbhtml;document.getElementById(\'trlbls\').innerHTML=tlhtml}function esc(s){return String(s).replace(/&/g,\'&amp;\').replace(/</g,\'&lt;\').replace(/>/g,\'&gt;\').replace(/"/g,\'&quot;\')}function fd(s){if(!s)return\'\';try{var d=new Date(s);return d.toLocaleDateString(\'en-GB\',{day:\'numeric\',month:\'short\'})}catch(e){return s}}function fdf(s){if(!s)return\'\';try{return new Date(s).toLocaleString(\'en-GB\')}catch(e){return s}}function toast(msg,err){var t=document.createElement(\'div\');t.className=\'toast\'+(err?\' err\':\'\');t.textContent=msg;document.body.appendChild(t);setTimeout(function(){t.remove()},2100)}</script>\n</body></html>'
_F = _T.replace('__JSONDATA__', _DATA)
_url = 'data:text/html;base64,' + _b64.b64encode(_F.encode('utf-8')).decode('ascii')

display(_IHTML(f"""
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@800&family=JetBrains+Mono&display=swap" rel="stylesheet"/>
<div style="font-family:Syne,sans-serif;background:#090d12;border:1px solid #1d2b3a;border-radius:10px;padding:16px 22px;display:flex;align-items:center;justify-content:space-between;gap:14px;flex-wrap:wrap">
  <div style="display:flex;align-items:center;gap:10px">
    <span style="font-size:20px;filter:drop-shadow(0 0 8px #00c17c80)">&#x1F99E;</span>
    <div>
      <div style="font-size:13px;font-weight:800;color:#dce8f2">OpenClaw <span style="color:#00c17c">x</span> ArangoDB</div>
      <div style="font-size:9px;color:#3d5068;font-family:JetBrains Mono,monospace;letter-spacing:.1em">DIGITAL BRAIN</div>
    </div>
  </div>
  <div style="display:flex;gap:16px">
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_st["memories"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Memories</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_st["entities"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Entities</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_st["entity_edges"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Edges</div></div>
    <div style="text-align:center"><div style="font-family:JetBrains Mono,monospace;font-size:19px;color:#00c17c">{_st["sessions"]}</div><div style="font-size:9px;color:#3d5068;text-transform:uppercase;letter-spacing:.08em">Sessions</div></div>
  </div>
  <a href="{_url}" target="_blank" style="padding:11px 20px;background:#00c17c;color:#000;border-radius:6px;font-family:Syne,sans-serif;font-size:11px;font-weight:800;letter-spacing:.06em;text-transform:uppercase;text-decoration:none;box-shadow:0 0 14px #00c17c40;white-space:nowrap">
    Open Brain UI &#8594;
  </a>
</div>"""))